In [1]:
import pandas as pd
from surprise import Dataset
from surprise import Reader
from surprise import SVD
from surprise.model_selection import train_test_split
from surprise.accuracy import rmse

In [2]:
from surprise import SVD

model = SVD()
print("Surprise is working!")

Surprise is working!


In [ ]:
ratings = pd.read_csv(../data/ml-100k/ml-100k/u.data
    "",
    sep=r"\s+",
    header=None,
    names=["user_id", "movie_id", "rating", "timestamp"]
)

In [11]:
ratings.head()


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [12]:
ratings.shape

(100000, 4)

In [13]:
print(ratings.columns.tolist())

['user_id', 'movie_id', 'rating', 'timestamp']


In [14]:
from surprise import Dataset, Reader

reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(
    ratings[['user_id', 'movie_id', 'rating']],
    reader
)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print("Trainset created successfully!")

Trainset created successfully!


In [15]:
from surprise import SVD
from surprise import accuracy

model = SVD(random_state=42)

model.fit(trainset)

print("Model trained successfully!")

Model trained successfully!


In [16]:
predictions = model.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

print("RMSE:", rmse)
print("MAE :", mae)

RMSE: 0.9352
MAE:  0.7375
RMSE: 0.935171451026933
MAE : 0.7375092386530285


In [17]:
user_id = 196
movie_id = 242

prediction = model.predict(user_id, movie_id)

print(prediction)
print("Predicted Rating:", round(prediction.est, 2))

user: 196        item: 242        r_ui = None   est = 3.82   {'was_impossible': False}
Predicted Rating: 3.82


In [18]:
movies = pd.read_csv(
    "../data/ml-100k/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["movie_id", "title"]
)

movies.head()

,movie_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [19]:
# Choose a user
user_id = 196

# Movies already rated by this user
watched_movies = ratings[ratings['user_id'] == user_id]['movie_id'].tolist()

print("User has rated", len(watched_movies), "movies")
print("First 10 movie IDs:", watched_movies[:10])

User has rated 39 movies
First 10 movie IDs: [242, 393, 381, 251, 655, 67, 306, 238, 663, 111]


In [20]:
# Get all movie IDs
all_movies = movies['movie_id'].tolist()

# Movies not yet rated by the user
unwatched_movies = [
    movie for movie in all_movies
    if movie not in watched_movies
]

print("Total movies:", len(all_movies))
print("Watched movies:", len(watched_movies))
print("Unwatched movies:", len(unwatched_movies))

Total movies: 1682
Watched movies: 39
Unwatched movies: 1643


In [21]:
# Predict ratings for all unseen movies

predictions = []

for movie_id in unwatched_movies:
    predicted_rating = model.predict(user_id, movie_id).est

    predictions.append((movie_id, predicted_rating))

print("Total predictions made:", len(predictions))
predictions[:5]

Total predictions made: 1643


[(1, np.float64(3.8811834350613434)),
 (2, np.float64(3.2890875456143336)),
 (3, np.float64(3.133752041322595)),
 (4, np.float64(3.7496587866438325)),
 (5, np.float64(3.3173869995806107))]

In [22]:
# Sort movies by predicted rating (highest first)

predictions.sort(key=lambda x: x[1], reverse=True)

predictions[:10]

[(318, np.float64(4.643424313713487)),
 (64, np.float64(4.605961892294713)),
 (474, np.float64(4.578762463473864)),
 (408, np.float64(4.555357511409159)),
 (357, np.float64(4.537785264491893)),
 (480, np.float64(4.512972284119157)),
 (511, np.float64(4.500572169683164)),
 (603, np.float64(4.4856823712198075)),
 (100, np.float64(4.484789900487661)),
 (513, np.float64(4.449328913322934))]

In [23]:
# Select the Top 10 recommendations

top_10 = predictions[:10]

top_10

[(318, np.float64(4.643424313713487)),
 (64, np.float64(4.605961892294713)),
 (474, np.float64(4.578762463473864)),
 (408, np.float64(4.555357511409159)),
 (357, np.float64(4.537785264491893)),
 (480, np.float64(4.512972284119157)),
 (511, np.float64(4.500572169683164)),
 (603, np.float64(4.4856823712198075)),
 (100, np.float64(4.484789900487661)),
 (513, np.float64(4.449328913322934))]

In [24]:
print("🎬 Top 10 Movie Recommendations\n")

for movie_id, predicted_rating in top_10:
    
    title = movies[movies["movie_id"] == movie_id]["title"].values[0]
    
    print(f"{title} ⭐ {predicted_rating:.2f}")

🎬 Top 10 Movie Recommendations

Schindler's List (1993) ⭐ 4.64
Shawshank Redemption, The (1994) ⭐ 4.61
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963) ⭐ 4.58
Close Shave, A (1995) ⭐ 4.56
One Flew Over the Cuckoo's Nest (1975) ⭐ 4.54
North by Northwest (1959) ⭐ 4.51
Lawrence of Arabia (1962) ⭐ 4.50
Rear Window (1954) ⭐ 4.49
Fargo (1996) ⭐ 4.48
Third Man, The (1949) ⭐ 4.45


In [25]:
# Count how many ratings each movie received

movie_stats = ratings.groupby("movie_id").agg(
    average_rating=("rating", "mean"),
    rating_count=("rating", "count")
)

movie_stats.head()

,average_rating,rating_count
movie_id,,
1,3.878319,452
2,3.206107,131
3,3.033333,90
4,3.550239,209
5,3.302326,86


In [26]:
# Merge movie statistics with movie titles

popularity_df = movie_stats.merge(
    movies,
    on="movie_id"
)

popularity_df.head()

,movie_id,average_rating,rating_count,title
0,1,3.878319,452,Toy Story (1995)
1,2,3.206107,131,GoldenEye (1995)
2,3,3.033333,90,Four Rooms (1995)
3,4,3.550239,209,Get Shorty (1995)
4,5,3.302326,86,Copycat (1995)


In [27]:
# Top 10 most popular movies

top_popular = popularity_df.sort_values(
    by=["average_rating", "rating_count"],
    ascending=False
)

top_popular[
    ["title", "average_rating", "rating_count"]
].head(10)

,title,average_rating,rating_count
1188,Prefontaine (1997),5.0,3
1292,Star Kid (1997),5.0,3
1466,"Saint of Fort Washington, The (1993)",5.0,2
1499,Santa with Muscles (1996),5.0,2
813,"Great Day in Harlem, A (1994)",5.0,1
1121,They Made Me a Criminal (1939),5.0,1
1200,Marlene Dietrich: Shadow and Light (1996),5.0,1
1535,Aiqing wansui (1994),5.0,1
1598,Someone Else's America (1995),5.0,1
1652,Entertaining Angels: The Dorothy Day Story (1996),5.0,1


In [28]:
# Keep only movies that have at least 50 ratings

popular_movies = popularity_df[popularity_df["rating_count"] >= 50]

top_popular = popular_movies.sort_values(
    by=["average_rating", "rating_count"],
    ascending=False
)

top_popular[
    ["title", "average_rating", "rating_count"]
].head(10)

,title,average_rating,rating_count
407,"Close Shave, A (1995)",4.491071,112
317,Schindler's List (1993),4.466443,298
168,"Wrong Trousers, The (1993)",4.466102,118
482,Casablanca (1942),4.456790,243
113,Wallace & Gromit: The Best of Aardman Animatio...,4.447761,67
63,"Shawshank Redemption, The (1994)",4.445230,283
602,Rear Window (1954),4.387560,209
11,"Usual Suspects, The (1995)",4.385768,267
49,Star Wars (1977),4.358491,583
177,12 Angry Men (1957),4.344000,125


In [29]:
# Create User-Movie Matrix

user_movie_matrix = ratings.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

user_movie_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
# Fill missing ratings with 0
user_movie_matrix_filled = user_movie_matrix.fillna(0)

print(user_movie_matrix_filled.shape)
user_movie_matrix_filled.head()

(943, 1682)


movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [31]:
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
# Calculate similarity between movies

movie_similarity = cosine_similarity(user_movie_matrix_filled.T)

print(movie_similarity.shape)

(1682, 1682)


In [33]:
movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=user_movie_matrix_filled.columns,
    columns=user_movie_matrix_filled.columns
)

movie_similarity_df.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
movie_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.402382,0.330245,0.454938,0.286714,0.116344,0.620979,0.481114,0.496288,0.273935,...,0.035387,0.0,0.000000,0.000000,0.035387,0.0,0.0,0.0,0.047183,0.047183
2,0.402382,1.000000,0.273069,0.502571,0.318836,0.083563,0.383403,0.337002,0.255252,0.171082,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.078299,0.078299
3,0.330245,0.273069,1.000000,0.324866,0.212957,0.106722,0.372921,0.200794,0.273669,0.158104,...,0.000000,0.0,0.000000,0.000000,0.032292,0.0,0.0,0.0,0.000000,0.096875
4,0.454938,0.502571,0.324866,1.000000,0.334239,0.090308,0.489283,0.490236,0.419044,0.252561,...,0.000000,0.0,0.094022,0.094022,0.037609,0.0,0.0,0.0,0.056413,0.075218
5,0.286714,0.318836,0.212957,0.334239,1.000000,0.037299,0.334769,0.259161,0.272448,0.055453,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.094211


In [34]:
def recommend_similar_movies(movie_title, n=10):
    # Find movie ID
    movie = movies[movies['title'] == movie_title]

    if movie.empty:
        print("Movie not found!")
        return

    movie_id = movie.iloc[0]['movie_id']

    # Get similarity scores
    similar_scores = movie_similarity_df[movie_id].sort_values(ascending=False)

    # Remove the movie itself
    similar_scores = similar_scores.iloc[1:n+1]

    # Get movie titles
    recommendations = movies[
        movies['movie_id'].isin(similar_scores.index)
    ][['movie_id', 'title']]

    recommendations = recommendations.merge(
        similar_scores.rename("similarity"),
        left_on="movie_id",
        right_index=True
    )

    recommendations = recommendations.sort_values(
        by="similarity",
        ascending=False
    )

    return recommendations[['title', 'similarity']]

In [35]:
recommend_similar_movies("Toy Story (1995)")

,title,similarity
49,Star Wars (1977),0.734572
180,Return of the Jedi (1983),0.699925
120,Independence Day (ID4) (1996),0.689786
116,"Rock, The (1996)",0.664555
404,Mission: Impossible (1996),0.641322
150,Willy Wonka and the Chocolate Factory (1971),0.638158
221,Star Trek: First Contact (1996),0.636727
99,Fargo (1996),0.630601
236,Jerry Maguire (1996),0.624075
173,Raiders of the Lost Ark (1981),0.622382


In [36]:
def search_movie(movie_name):
    results = movies[
        movies["title"].str.contains(movie_name, case=False, na=False)
    ][["movie_id", "title"]]

    return results.head(10)

In [37]:
search_movie("toy")

,movie_id,title
0,1,Toy Story (1995)


In [38]:
search_movie("mission")

,movie_id,title
404,405,Mission: Impossible (1996)


In [39]:
def recommend(movie_name):
    # Search for the movie
    result = search_movie(movie_name)

    if result.empty:
        print("❌ Movie not found!")
        return

    # Take the first matching movie
    selected_movie = result.iloc[0]["title"]

    print(f"🎬 Selected Movie: {selected_movie}")
    print("\nRecommended Movies:\n")

    recommendations = recommend_similar_movies(selected_movie)

    display(recommendations)

In [40]:
recommend("toy")

🎬 Selected Movie: Toy Story (1995)

Recommended Movies:



,title,similarity
49,Star Wars (1977),0.734572
180,Return of the Jedi (1983),0.699925
120,Independence Day (ID4) (1996),0.689786
116,"Rock, The (1996)",0.664555
404,Mission: Impossible (1996),0.641322
150,Willy Wonka and the Chocolate Factory (1971),0.638158
221,Star Trek: First Contact (1996),0.636727
99,Fargo (1996),0.630601
236,Jerry Maguire (1996),0.624075
173,Raiders of the Lost Ark (1981),0.622382


In [41]:
# Movies with the fewest ratings

least_rated = popularity_df.sort_values(
    by="rating_count",
    ascending=True
)

least_rated[
    ["title", "rating_count", "average_rating"]
].head(20)

,title,rating_count,average_rating
1672,Mirage (1995),1,3.0
1681,Scream of Stone (Schrei aus Stein) (1991),1,3.0
1670,"Further Gesture, A (1996)",1,1.0
1669,Tainted (1998),1,3.0
1668,MURDER and murder (1996),1,2.0
1667,Wedding Bell Blues (1996),1,3.0
1636,Girls Town (1996),1,3.0
1635,Brothers in Trouble (1995),1,4.0
1634,Two Friends (1986),1,3.0
1633,Etz Hadomim Tafus (Under the Domin Tree) (1994),1,2.0


In [42]:
# Count how many movies have very few ratings

rating_count_distribution = popularity_df["rating_count"].value_counts().sort_index()

print("Movies with exactly 1 rating:",
      rating_count_distribution.get(1, 0))

print("Movies with 2 or fewer ratings:",
      rating_count_distribution[rating_count_distribution.index <= 2].sum())

print("Movies with 5 or fewer ratings:",
      rating_count_distribution[rating_count_distribution.index <= 5].sum())

Movies with exactly 1 rating: 141
Movies with 2 or fewer ratings: 209
Movies with 5 or fewer ratings: 384


In [43]:
# Pick a movie with exactly 1 rating

rare_movie = least_rated[
    least_rated["rating_count"] == 1
].iloc[0]["title"]

print("Rare movie:", rare_movie)

print("\nRecommendations:")
recommend_similar_movies(rare_movie)

Rare movie: Mirage (1995)

Recommendations:


,title,similarity
1277,Selena (1997),0.326860
542,"Misérables, Les (1995)",0.286770
1152,Backbeat (1993),0.255031
1044,Fearless (1993),0.237775
609,Gigi (1958),0.205499
370,"Bridges of Madison County, The (1995)",0.178800
611,Lost Horizon (1937),0.170561
631,Sophie's Choice (1982),0.163956
1062,"Little Princess, A (1995)",0.163572
498,Cat on a Hot Tin Roof (1958),0.158035


In [44]:
# Pick a highly rated/popular movie

popular_movie = popularity_df.sort_values(
    by="rating_count",
    ascending=False
).iloc[0]["title"]

print("Popular movie:", popular_movie)

print("\nRecommendations:")
recommend_similar_movies(popular_movie)

Popular movie: Star Wars (1977)

Recommendations:


,title,similarity
180,Return of the Jedi (1983),0.884476
173,Raiders of the Lost Ark (1981),0.764885
171,"Empire Strikes Back, The (1980)",0.749819
0,Toy Story (1995),0.734572
126,"Godfather, The (1972)",0.697332
120,Independence Day (ID4) (1996),0.692837
209,Indiana Jones and the Last Crusade (1989),0.689343
99,Fargo (1996),0.686533
97,"Silence of the Lambs, The (1991)",0.676428
221,Star Trek: First Contact (1996),0.673975


In [45]:
import numpy as np

print("Users:", ratings["user_id"].nunique())
print("Movies:", ratings["movie_id"].nunique())
print("Ratings:", len(ratings))

Users: 943
Movies: 1682
Ratings: 100000


In [46]:
# Create User-Movie Rating Matrix

rating_matrix = ratings.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

print("Rating matrix shape:", rating_matrix.shape)

Rating matrix shape: (943, 1682)


In [47]:
# Fill missing ratings with 0
rating_matrix_filled = rating_matrix.fillna(0)

print("Matrix shape:", rating_matrix_filled.shape)
print("Missing values:", rating_matrix_filled.isna().sum().sum())

Matrix shape: (943, 1682)
Missing values: 0


In [48]:
# Matrix Factorization parameters

num_users = rating_matrix_filled.shape[0]
num_movies = rating_matrix_filled.shape[1]

num_factors = 20

np.random.seed(42)

# User latent-factor matrix
user_factors = np.random.normal(
    0,
    0.1,
    (num_users, num_factors)
)

# Movie latent-factor matrix
movie_factors = np.random.normal(
    0,
    0.1,
    (num_movies, num_factors)
)

print("User factors shape:", user_factors.shape)
print("Movie factors shape:", movie_factors.shape)

User factors shape: (943, 20)
Movie factors shape: (1682, 20)


In [49]:
# Generate predicted ratings

predicted_ratings = np.dot(
    user_factors,
    movie_factors.T
)

print("Predicted ratings shape:", predicted_ratings.shape)
print("Sample predicted ratings:")
print(predicted_ratings[:5, :5])

Predicted ratings shape: (943, 1682)
Sample predicted ratings:
[[-0.00196172 -0.05311089  0.01672943 -0.08575642 -0.02480183]
 [-0.02568675 -0.01714871  0.00729992  0.01058088  0.00973872]
 [ 0.00987684 -0.02463667 -0.02638981  0.0356014  -0.0069652 ]
 [ 0.02014932  0.02956399 -0.07200099  0.00894768  0.03592534]
 [-0.03471643  0.04709903  0.00040014 -0.01836855 -0.00477987]]


In [50]:
# Create a mask for observed ratings

rating_mask = (rating_matrix_filled.values > 0).astype(float)

print("Observed ratings:", int(rating_mask.sum()))
print("Total matrix entries:", rating_mask.size)
print("Sparsity:", 1 - rating_mask.mean())

Observed ratings: 100000
Total matrix entries: 1586126
Sparsity: 0.9369533063577546


In [51]:
# Calculate prediction error only for observed ratings

actual_ratings = rating_matrix_filled.values

errors = (actual_ratings - predicted_ratings) * rating_mask

rmse = np.sqrt(
    np.sum(errors ** 2) / np.sum(rating_mask)
)

print("Initial RMSE:", rmse)

Initial RMSE: 3.7051209932938005


In [52]:
# Matrix Factorization using Gradient Descent

learning_rate = 0.005
reg = 0.02
epochs = 20

actual_ratings = rating_matrix_filled.values

for epoch in range(epochs):

    for user in range(num_users):
        for movie in range(num_movies):

            # Skip unrated movies
            if rating_mask[user, movie] == 0:
                continue

            # Actual rating
            actual = actual_ratings[user, movie]

            # Predicted rating
            predicted = np.dot(
                user_factors[user],
                movie_factors[movie]
            )

            # Prediction error
            error = actual - predicted

            # Save current factors before updating
            user_factor_old = user_factors[user].copy()
            movie_factor_old = movie_factors[movie].copy()

            # Gradient descent updates
            user_factors[user] += learning_rate * (
                error * movie_factor_old
                - reg * user_factor_old
            )

            movie_factors[movie] += learning_rate * (
                error * user_factor_old
                - reg * movie_factor_old
            )

    # Calculate RMSE after each epoch
    predicted_ratings = np.dot(
        user_factors,
        movie_factors.T
    )

    errors = (
        actual_ratings - predicted_ratings
    ) * rating_mask

    epoch_rmse = np.sqrt(
        np.sum(errors ** 2) / np.sum(rating_mask)
    )

    print(f"Epoch {epoch + 1}/{epochs} - RMSE: {epoch_rmse:.4f}")

Epoch 1/20 - RMSE: 3.2973
Epoch 2/20 - RMSE: 1.6265
Epoch 3/20 - RMSE: 1.1952
Epoch 4/20 - RMSE: 1.0629
Epoch 5/20 - RMSE: 1.0050
Epoch 6/20 - RMSE: 0.9738
Epoch 7/20 - RMSE: 0.9540
Epoch 8/20 - RMSE: 0.9397
Epoch 9/20 - RMSE: 0.9280
Epoch 10/20 - RMSE: 0.9175
Epoch 11/20 - RMSE: 0.9077
Epoch 12/20 - RMSE: 0.8983
Epoch 13/20 - RMSE: 0.8890
Epoch 14/20 - RMSE: 0.8799
Epoch 15/20 - RMSE: 0.8710
Epoch 16/20 - RMSE: 0.8622
Epoch 17/20 - RMSE: 0.8534
Epoch 18/20 - RMSE: 0.8448
Epoch 19/20 - RMSE: 0.8361
Epoch 20/20 - RMSE: 0.8276


In [53]:
from sklearn.model_selection import train_test_split

train_ratings, test_ratings = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

print("Training ratings:", len(train_ratings))
print("Testing ratings:", len(test_ratings))

Training ratings: 80000
Testing ratings: 20000


In [54]:
# Create fresh latent-factor matrices for the train/test experiment

num_factors = 20

np.random.seed(42)

user_factors_scratch = np.random.normal(
    0,
    0.1,
    (num_users, num_factors)
)

movie_factors_scratch = np.random.normal(
    0,
    0.1,
    (num_movies, num_factors)
)

print("User factors:", user_factors_scratch.shape)
print("Movie factors:", movie_factors_scratch.shape)

User factors: (943, 20)
Movie factors: (1682, 20)


In [55]:
# Hyperparameters
learning_rate = 0.005
reg = 0.02
epochs = 20

for epoch in range(epochs):

    # Shuffle training data each epoch
    train_epoch = train_ratings.sample(
        frac=1,
        random_state=42 + epoch
    )

    for _, row in train_epoch.iterrows():

        user_id = int(row["user_id"])
        movie_id = int(row["movie_id"])
        actual_rating = row["rating"]

        # Convert IDs to matrix indices
        user_idx = user_id - 1
        movie_idx = movie_id - 1

        # Current prediction
        predicted_rating = np.dot(
            user_factors_scratch[user_idx],
            movie_factors_scratch[movie_idx]
        )

        # Prediction error
        error = actual_rating - predicted_rating

        # Save current factors before updating
        user_old = user_factors_scratch[user_idx].copy()
        movie_old = movie_factors_scratch[movie_idx].copy()

        # Update user factors
        user_factors_scratch[user_idx] += learning_rate * (
            error * movie_old
            - reg * user_old
        )

        # Update movie factors
        movie_factors_scratch[movie_idx] += learning_rate * (
            error * user_old
            - reg * movie_old
        )

    # Calculate training RMSE
    train_predictions = []

    for _, row in train_ratings.iterrows():

        user_idx = int(row["user_id"]) - 1
        movie_idx = int(row["movie_id"]) - 1

        prediction = np.dot(
            user_factors_scratch[user_idx],
            movie_factors_scratch[movie_idx]
        )

        train_predictions.append(prediction)

    train_actual = train_ratings["rating"].values

    train_rmse = np.sqrt(
        np.mean(
            (train_actual - np.array(train_predictions)) ** 2
        )
    )

    print(
        f"Epoch {epoch + 1}/{epochs} - "
        f"Training RMSE: {train_rmse:.4f}"
    )

Epoch 1/20 - Training RMSE: 3.6693
Epoch 2/20 - Training RMSE: 2.4206
Epoch 3/20 - Training RMSE: 1.4992
Epoch 4/20 - Training RMSE: 1.2086
Epoch 5/20 - Training RMSE: 1.0841
Epoch 6/20 - Training RMSE: 1.0202
Epoch 7/20 - Training RMSE: 0.9825
Epoch 8/20 - Training RMSE: 0.9577
Epoch 9/20 - Training RMSE: 0.9401
Epoch 10/20 - Training RMSE: 0.9259
Epoch 11/20 - Training RMSE: 0.9134
Epoch 12/20 - Training RMSE: 0.9025
Epoch 13/20 - Training RMSE: 0.8922
Epoch 14/20 - Training RMSE: 0.8826
Epoch 15/20 - Training RMSE: 0.8733
Epoch 16/20 - Training RMSE: 0.8649
Epoch 17/20 - Training RMSE: 0.8561
Epoch 18/20 - Training RMSE: 0.8474
Epoch 19/20 - Training RMSE: 0.8387
Epoch 20/20 - Training RMSE: 0.8306


In [56]:
# Evaluate scratch model on unseen test ratings

test_predictions = []

for _, row in test_ratings.iterrows():

    user_idx = int(row["user_id"]) - 1
    movie_idx = int(row["movie_id"]) - 1

    prediction = np.dot(
        user_factors_scratch[user_idx],
        movie_factors_scratch[movie_idx]
    )

    # Keep predictions within the valid rating range
    prediction = np.clip(prediction, 1, 5)

    test_predictions.append(prediction)

test_actual = test_ratings["rating"].values

test_rmse = np.sqrt(
    np.mean(
        (test_actual - np.array(test_predictions)) ** 2
    )
)

print("Scratch Model Test RMSE:", round(test_rmse, 4))

Scratch Model Test RMSE: 0.9402


In [57]:
def train_matrix_factorization(
    train_data,
    num_users,
    num_movies,
    num_factors=20,
    learning_rate=0.005,
    reg=0.02,
    epochs=20
):
    # Initialize latent factors
    np.random.seed(42)

    user_factors = np.random.normal(
        0,
        0.1,
        (num_users, num_factors)
    )

    movie_factors = np.random.normal(
        0,
        0.1,
        (num_movies, num_factors)
    )

    # Training
    for epoch in range(epochs):

        train_epoch = train_data.sample(
            frac=1,
            random_state=42 + epoch
        )

        for _, row in train_epoch.iterrows():

            user_idx = int(row["user_id"]) - 1
            movie_idx = int(row["movie_id"]) - 1
            actual_rating = row["rating"]

            # Prediction
            prediction = np.dot(
                user_factors[user_idx],
                movie_factors[movie_idx]
            )

            # Error
            error = actual_rating - prediction

            # Save old factors
            user_old = user_factors[user_idx].copy()
            movie_old = movie_factors[movie_idx].copy()

            # Update user factors
            user_factors[user_idx] += learning_rate * (
                error * movie_old
                - reg * user_old
            )

            # Update movie factors
            movie_factors[movie_idx] += learning_rate * (
                error * user_old
                - reg * movie_old
            )

    return user_factors, movie_factors

In [58]:
def evaluate_matrix_factorization(
    user_factors,
    movie_factors,
    test_data
):
    predictions = []
    actuals = []

    for _, row in test_data.iterrows():

        user_idx = int(row["user_id"]) - 1
        movie_idx = int(row["movie_id"]) - 1

        prediction = np.dot(
            user_factors[user_idx],
            movie_factors[movie_idx]
        )

        prediction = np.clip(prediction, 1, 5)

        predictions.append(prediction)
        actuals.append(row["rating"])

    rmse = np.sqrt(
        np.mean(
            (np.array(actuals) - np.array(predictions)) ** 2
        )
    )

    return rmse

In [59]:
factor_results = []

for factors in [10, 20, 50]:

    print(f"\nTraining with {factors} latent factors...")

    user_factors_test, movie_factors_test = train_matrix_factorization(
        train_data=train_ratings,
        num_users=num_users,
        num_movies=num_movies,
        num_factors=factors,
        learning_rate=0.005,
        reg=0.02,
        epochs=20
    )

    rmse = evaluate_matrix_factorization(
        user_factors_test,
        movie_factors_test,
        test_ratings
    )

    factor_results.append({
        "factors": factors,
        "test_rmse": rmse
    })

    print(f"Test RMSE: {rmse:.4f}")


Training with 10 latent factors...
Test RMSE: 0.9419

Training with 20 latent factors...
Test RMSE: 0.9402

Training with 50 latent factors...
Test RMSE: 0.9433


In [60]:
factor_results_df = pd.DataFrame(factor_results)

factor_results_df

,factors,test_rmse
0,10,0.941877
1,20,0.940234
2,50,0.943295


In [61]:
learning_rate_results = []

for lr in [0.001, 0.005, 0.01]:

    print(f"\nTraining with learning rate = {lr}...")

    user_factors_test, movie_factors_test = train_matrix_factorization(
        train_data=train_ratings,
        num_users=num_users,
        num_movies=num_movies,
        num_factors=20,
        learning_rate=lr,
        reg=0.02,
        epochs=20
    )

    rmse = evaluate_matrix_factorization(
        user_factors_test,
        movie_factors_test,
        test_ratings
    )

    learning_rate_results.append({
        "learning_rate": lr,
        "test_rmse": rmse
    })

    print(f"Test RMSE: {rmse:.4f}")


Training with learning rate = 0.001...
Test RMSE: 1.2167

Training with learning rate = 0.005...
Test RMSE: 0.9402

Training with learning rate = 0.01...
Test RMSE: 0.9460


In [62]:
learning_rate_results_df = pd.DataFrame(
    learning_rate_results
)

learning_rate_results_df

,learning_rate,test_rmse
0,0.001,1.216693
1,0.005,0.940234
2,0.010,0.945954


In [63]:
regularization_results = []

for reg in [0.01, 0.02, 0.05]:

    print(f"\nTraining with regularization = {reg}...")

    user_factors_test, movie_factors_test = train_matrix_factorization(
        train_data=train_ratings,
        num_users=num_users,
        num_movies=num_movies,
        num_factors=20,
        learning_rate=0.005,
        reg=reg,
        epochs=20
    )

    rmse = evaluate_matrix_factorization(
        user_factors_test,
        movie_factors_test,
        test_ratings
    )

    regularization_results.append({
        "regularization": reg,
        "test_rmse": rmse
    })

    print(f"Test RMSE: {rmse:.4f}")


Training with regularization = 0.01...
Test RMSE: 0.9421

Training with regularization = 0.02...
Test RMSE: 0.9402

Training with regularization = 0.05...
Test RMSE: 0.9410


In [64]:
regularization_results_df = pd.DataFrame(
    regularization_results
)

regularization_results_df

,regularization,test_rmse
0,0.01,0.942070
1,0.02,0.940234
2,0.05,0.941027


In [65]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

In [66]:
# Prepare the same training data for Surprise

reader = Reader(rating_scale=(1, 5))

surprise_train = Dataset.load_from_df(
    train_ratings[["user_id", "movie_id", "rating"]],
    reader
)

surprise_trainset = surprise_train.build_full_trainset()

print("Surprise training ratings:", surprise_trainset.n_ratings)

Surprise training ratings: 80000


In [67]:
# Train Surprise SVD

surprise_model = SVD(
    n_factors=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

surprise_model.fit(surprise_trainset)

print("Surprise SVD training completed.")

Surprise SVD training completed.


In [68]:
# Evaluate Surprise SVD on the same 20,000 test ratings

surprise_predictions = []

for _, row in test_ratings.iterrows():

    prediction = surprise_model.predict(
        int(row["user_id"]),
        int(row["movie_id"])
    ).est

    surprise_predictions.append(prediction)

test_actual = test_ratings["rating"].values

surprise_rmse = np.sqrt(
    np.mean(
        (test_actual - np.array(surprise_predictions)) ** 2
    )
)

print("Surprise SVD Test RMSE:", round(surprise_rmse, 4))

Surprise SVD Test RMSE: 0.9311


In [69]:
comparison_results = pd.DataFrame({
    "Model": [
        "Scratch Matrix Factorization",
        "Surprise SVD"
    ],
    "Test RMSE": [
        test_rmse,
        surprise_rmse
    ]
})

comparison_results

,Model,Test RMSE
0,Scratch Matrix Factorization,0.940234
1,Surprise SVD,0.931124


In [70]:
improvement = (
    (test_rmse - surprise_rmse)
    / test_rmse
) * 100

print(
    f"Surprise SVD improvement: {improvement:.2f}%"
)

Surprise SVD improvement: 0.97%


In [71]:
# Simulate a completely new user

new_user_id = 1000

try:
    prediction = surprise_model.predict(
        new_user_id,
        1
    )

    print("Predicted rating:", prediction.est)

except Exception as e:
    print("Error:", e)

Predicted rating: 3.935397545080085


In [72]:
# Test multiple movies for the completely new user

new_user_id = 1000

for movie_id in [1, 50, 100, 200, 500]:

    prediction = surprise_model.predict(
        new_user_id,
        movie_id
    )

    print(
        f"Movie {movie_id}: "
        f"{prediction.est:.4f}"
    )

Movie 1: 3.9354
Movie 50: 4.4249
Movie 100: 4.1512
Movie 200: 3.8990
Movie 500: 3.8675


In [73]:
def cold_start_recommendations(n=10):
    
    recommendations = top_popular[
        ["title", "average_rating", "rating_count"]
    ].head(n)

    return recommendations

In [74]:
cold_start_recommendations()

,title,average_rating,rating_count
407,"Close Shave, A (1995)",4.491071,112
317,Schindler's List (1993),4.466443,298
168,"Wrong Trousers, The (1993)",4.466102,118
482,Casablanca (1942),4.456790,243
113,Wallace & Gromit: The Best of Aardman Animatio...,4.447761,67
63,"Shawshank Redemption, The (1994)",4.445230,283
602,Rear Window (1954),4.387560,209
11,"Usual Suspects, The (1995)",4.385768,267
49,Star Wars (1977),4.358491,583
177,12 Angry Men (1957),4.344000,125


In [75]:
# Ranking evaluation settings

TOP_N = 10
RELEVANT_THRESHOLD = 4.0

print("Top-N:", TOP_N)
print("Relevant rating threshold:", RELEVANT_THRESHOLD)

Top-N: 10
Relevant rating threshold: 4.0


In [ ]:
def get_top_n_recommendations(user_id, n=10):

    # Only exclude movies the user rated in TRAINING data
    rated_movies = set(
        train_ratings[
            train_ratings["user_id"] == user_id
        ]["movie_id"]
    )

    all_movies = ratings["movie_id"].unique()

    predictions = []

    for movie_id in all_movies:

        # Don't recommend movies already seen during training
        if movie_id not in rated_movies:

            prediction = surprise_model.predict(
                user_id,
                movie_id
            )

            predictions.append(
                (movie_id, prediction.est)
            )

    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return predictions[:n]

In [84]:
top_10 = get_top_n_recommendations(196, 10)

top_10

[(np.int64(408), np.float64(4.523982744146186)),
 (np.int64(318), np.float64(4.486446286285099)),
 (np.int64(64), np.float64(4.483954713802624)),
 (np.int64(178), np.float64(4.438692778997661)),
 (np.int64(114), np.float64(4.394478416136636)),
 (np.int64(603), np.float64(4.38319435083813)),
 (np.int64(483), np.float64(4.378216711126469)),
 (np.int64(169), np.float64(4.331429081557211)),
 (np.int64(513), np.float64(4.328384243155413)),
 (np.int64(320), np.float64(4.324581451906768))]

In [85]:
# Load movie titles

movies = pd.read_csv(
    "../data/ml-100k/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["movie_id", "title"]
)

movies.head()

,movie_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [81]:
# Display Top-10 recommendations with movie titles

top_10_df = pd.DataFrame(
    top_10,
    columns=["movie_id", "predicted_rating"]
)

top_10_df = top_10_df.merge(
    movies,
    on="movie_id",
    how="left"
)

top_10_df = top_10_df[
    ["movie_id", "title", "predicted_rating"]
]

top_10_df

,movie_id,title,predicted_rating
0,408,"Close Shave, A (1995)",4.523983
1,318,Schindler's List (1993),4.486446
2,64,"Shawshank Redemption, The (1994)",4.483955
3,178,12 Angry Men (1957),4.438693
4,114,Wallace & Gromit: The Best of Aardman Animatio...,4.394478
5,603,Rear Window (1954),4.383194
6,483,Casablanca (1942),4.378217
7,169,"Wrong Trousers, The (1993)",4.331429
8,513,"Third Man, The (1949)",4.328384
9,320,Paradise Lost: The Child Murders at Robin Hood...,4.324581


In [82]:
# Get relevant movies for User 196 from the test set

user_id = 196

relevant_movies = set(
    test_ratings[
        (test_ratings["user_id"] == user_id) &
        (test_ratings["rating"] >= RELEVANT_THRESHOLD)
    ]["movie_id"]
)

recommended_movies = set(
    top_10_df["movie_id"]
)

# Movies that were both recommended and actually relevant
relevant_recommendations = (
    recommended_movies & relevant_movies
)

precision_at_10 = (
    len(relevant_recommendations) / TOP_N
)

recall_at_10 = (
    len(relevant_recommendations) / len(relevant_movies)
    if len(relevant_movies) > 0
    else 0
)

print("User:", user_id)
print("Relevant movies in test set:", len(relevant_movies))
print("Relevant recommendations:", len(relevant_recommendations))
print("Precision@10:", round(precision_at_10, 4))
print("Recall@10:", round(recall_at_10, 4))

User: 196
Relevant movies in test set: 7
Relevant recommendations: 0
Precision@10: 0.0
Recall@10: 0.0


In [83]:
def get_top_n_recommendations(user_id, n=10):

    # Only exclude movies the user rated in TRAINING data
    rated_movies = set(
        train_ratings[
            train_ratings["user_id"] == user_id
        ]["movie_id"]
    )

    all_movies = ratings["movie_id"].unique()

    predictions = []

    for movie_id in all_movies:

        # Don't recommend movies already seen during training
        if movie_id not in rated_movies:

            prediction = surprise_model.predict(
                user_id,
                movie_id
            )

            predictions.append(
                (movie_id, prediction.est)
            )

    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return predictions[:n]

In [86]:
top_10 = get_top_n_recommendations(196, 10)

top_10_df = pd.DataFrame(
    top_10,
    columns=["movie_id", "predicted_rating"]
)

top_10_df = top_10_df.merge(
    movies,
    on="movie_id",
    how="left"
)

top_10_df = top_10_df[
    ["movie_id", "title", "predicted_rating"]
]

top_10_df

,movie_id,title,predicted_rating
0,408,"Close Shave, A (1995)",4.523983
1,318,Schindler's List (1993),4.486446
2,64,"Shawshank Redemption, The (1994)",4.483955
3,178,12 Angry Men (1957),4.438693
4,114,Wallace & Gromit: The Best of Aardman Animatio...,4.394478
5,603,Rear Window (1954),4.383194
6,483,Casablanca (1942),4.378217
7,169,"Wrong Trousers, The (1993)",4.331429
8,513,"Third Man, The (1949)",4.328384
9,320,Paradise Lost: The Child Murders at Robin Hood...,4.324581


In [87]:
user_id = 196

relevant_movies = set(
    test_ratings[
        (test_ratings["user_id"] == user_id) &
        (test_ratings["rating"] >= RELEVANT_THRESHOLD)
    ]["movie_id"]
)

recommended_movies = set(
    top_10_df["movie_id"]
)

relevant_recommendations = (
    recommended_movies & relevant_movies
)

precision_at_10 = (
    len(relevant_recommendations) / TOP_N
)

recall_at_10 = (
    len(relevant_recommendations) / len(relevant_movies)
    if len(relevant_movies) > 0
    else 0
)

print("User:", user_id)
print("Relevant movies in test set:", len(relevant_movies))
print("Relevant recommendations:", len(relevant_recommendations))
print("Precision@10:", round(precision_at_10, 4))
print("Recall@10:", round(recall_at_10, 4))

User: 196
Relevant movies in test set: 7
Relevant recommendations: 0
Precision@10: 0.0
Recall@10: 0.0


In [88]:
# Evaluate ranking performance across multiple users

evaluation_users = test_ratings["user_id"].unique()[:100]

precision_scores = []
recall_scores = []

for user_id in evaluation_users:

    # Relevant movies from the test set
    relevant_movies = set(
        test_ratings[
            (test_ratings["user_id"] == user_id) &
            (test_ratings["rating"] >= RELEVANT_THRESHOLD)
        ]["movie_id"]
    )

    # Skip users with no relevant test movies
    if len(relevant_movies) == 0:
        continue

    # Generate Top-10 recommendations
    user_top_n = get_top_n_recommendations(
        user_id,
        TOP_N
    )

    recommended_movies = {
        movie_id for movie_id, score in user_top_n
    }

    # Relevant recommended movies
    hits = (
        recommended_movies &
        relevant_movies
    )

    # Precision@10
    precision = len(hits) / TOP_N

    # Recall@10
    recall = len(hits) / len(relevant_movies)

    precision_scores.append(precision)
    recall_scores.append(recall)


mean_precision = np.mean(precision_scores)
mean_recall = np.mean(recall_scores)

print("Users evaluated:", len(precision_scores))
print("Mean Precision@10:", round(mean_precision, 4))
print("Mean Recall@10:", round(mean_recall, 4))

Users evaluated: 100
Mean Precision@10: 0.139
Mean Recall@10: 0.0567


In [89]:
final_results = pd.DataFrame({
    "Model": [
        "Scratch Matrix Factorization",
        "Surprise SVD"
    ],
    "Test RMSE": [
        test_rmse,
        surprise_rmse
    ],
    "Precision@10": [
        np.nan,
        mean_precision
    ],
    "Recall@10": [
        np.nan,
        mean_recall
    ]
})

final_results

,Model,Test RMSE,Precision@10,Recall@10
0,Scratch Matrix Factorization,0.940234,NaN,NaN
1,Surprise SVD,0.931124,0.139,0.056684
